## Pipeline

Tot nu toe is de code voor het opbouwen van een model vespreid over twee stappen. 
Een preprocesing stap waar alle transformaties op gebeuren op de data bruikbaar te maken zoals schaling, invullen van ontbrekende waarden, hogere orde features, ...
In een latere stap gebeurd dan het trainen van een model.
Echter is het zo dat als je het model in gebruik wilt nemen in een applicatie dat je ook exact dezelfde preprocessing stappen nodig hebt omdat je de data die je aan het model presenteert hetzelfde moet zijn.
Om deze reden is de vorige aanpak niet praktisch aangezien het eenvoudig is om een fout te maken.
Het is ook geen onderhoudbare oplossing omdat het veel manueel werk vereist indien er een bepaalde stap aangepakt wordt.

Om dit tegen te gaan kan men gebruik maken van Pipelines. 
Hiermee definieer je de flow die de data moet ondergaan om verwerkt te worden. 
De volledige pipeline of flow kan dan gebruikt worden om te trainen en te gebruiken in productie. 
Hierdoor worden de preprocessing stappen (zoals scalers, ordinal of one-hotencoder) steeds enkel op de training folds uitgevoerd.
Een bijkomende voordeel is dat ook parameters van de preprocessing stappen meegenomen kunnen worden in de gridsearch.

Let op dat NaN waarden invullen kan gebeuren in de pipeline maar rijen weglaten kan niet gedaan worden tijdens de pipeline. Echter is het gebruik van een pipeline een goede stap in het verhogen van de leesbaarheid van een code stuk en wordt het daardoor veelvuldig gebruikt.

In [1]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV

np.random.seed(0)

X, y = fetch_openml("titanic", version=1, as_frame=True, return_X_y=True)
X

,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,3,"Zabour, Miss. Hileni",female,14.5000,1,0,2665,14.4542,NaN,C,NaN,328.0,NaN
1305,3,"Zabour, Miss. Thamine",female,NaN,1,0,2665,14.4542,NaN,C,NaN,NaN,NaN
1306,3,"Zakarian, Mr. Mapriededer",male,26.5000,0,0,2656,7.2250,NaN,C,NaN,304.0,NaN
1307,3,"Zakarian, Mr. Ortin",male,27.0000,0,0,2670,7.2250,NaN,C,NaN,NaN,NaN


In [3]:
num_features = ['age', 'fare', 'parch', 'sibsp']
num_pipeline = Pipeline(steps=[
    ('imp', SimpleImputer(strategy='mean')), # vul null-waarden in  
    ('scaler', StandardScaler())
])

cat_features = ['sex', 'pclass', 'embarked']
cat_pipeline = Pipeline(steps=[
    ('imp', SimpleImputer(strategy='constant', fill_value='unknown')), # vul null-waarden in  
    ('scaler', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    # lijst met tuples met drie waarden
    # eerste is ook de naam
    # tweede wat er moet uitgevoerd worden
    # derde op welke kolommen het moet uitgevoerd worden
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
]) # gebruik , remainder='passthrough' als je kolommen die niet gebruikt worden toch door te laten

clf = Pipeline(steps=[
    # pipeline -> lijst met tuples met twee waarden
    # eerste is de naam
    # tweede is een pipeline/transformer/estimator
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression())
])

In [5]:
clf.fit(X, y)
clf.score(X,y)

0.787624140565317

In [16]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'preprocessor__num__imp__strategy': ['mean', 'median'], # dubble underscore bij het doorlopen van de keys van de pipeline
    'clf__C': range(1,10)
}

model = GridSearchCV(clf, param_grid, cv=5, scoring='precision_weighted', verbose=1)
model.fit(X,y)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


,estimator,Pipeline(step...egression())])
,param_grid,"{'clf__C': range(1, 10), 'preprocessor__num__imp__strategy': ['mean', 'median']}"
,scoring,'precision_weighted'
,n_jobs,None
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [13]:
model.cv_results_

{'mean_fit_time': array([0.01347399, 0.01088891, 0.01139188, 0.01288705, 0.01166706,
        0.0111393 , 0.01264386, 0.01044688, 0.01273093, 0.01068254,
        0.01195393, 0.01087036, 0.01055455, 0.01139698, 0.01164575,
        0.01142807, 0.00976191, 0.01310921]),
 'std_fit_time': array([0.00378943, 0.00252023, 0.00143755, 0.00063403, 0.00155354,
        0.00188476, 0.00076545, 0.00091353, 0.00075964, 0.00174513,
        0.00176525, 0.0015304 , 0.00119671, 0.00107659, 0.0014922 ,
        0.00144524, 0.001425  , 0.00180637]),
 'mean_score_time': array([0.00668745, 0.0052515 , 0.00675712, 0.00659285, 0.00644579,
        0.0063365 , 0.00696392, 0.0055757 , 0.00701833, 0.0056016 ,
        0.00734649, 0.00600886, 0.00672231, 0.00580716, 0.00663147,
        0.00621319, 0.00583267, 0.00703535]),
 'std_score_time': array([0.00211007, 0.00040362, 0.00030559, 0.00032815, 0.00128152,
        0.00075064, 0.00042261, 0.00104336, 0.0002736 , 0.00032875,
        0.00157303, 0.00066659, 0.00154152, 

## Oefening

Maak een pipeline voor een gridsearch uit te voeren op de diabetes-dataset in sklearn. Gebruik een Min-Max scaler voor de numerieke waarden en een ordinal encoder voor de categorieke kolommen.
Zoek naar de beste combinatie van hyperparameters van een SVM-classifier door middel van een grid search met cross validation.